In [1]:
from google.colab import drive, files
drive.mount('/content/drive')
# create a project folder in your Drive (change name if you want)
PROJECT_DIR = '/content/drive/MyDrive/ALS_project'
import os
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Project folder created at:", PROJECT_DIR)


ModuleNotFoundError: No module named 'google.colab'

In [ ]:
%cd /content/drive/MyDrive/ALS_project
!pwd


/content/drive/MyDrive/ALS_project
/content/drive/MyDrive/ALS_project


In [ ]:
!pip install -q numpy scipy librosa soundfile torch torchaudio scikit-learn matplotlib pandas tqdm


In [ ]:
import os

folders = [
    "data",          # for your audio dataset
    "results",       # trained model results
    "plots"          # graphs and confusion matrices
]

for f in folders:
    os.makedirs(f, exist_ok=True)

print("Folders created:", folders)
!ls


Folders created: ['data', 'results', 'plots']
data  plots  results


In [ ]:
%%writefile data_loader.py
import os
import csv
import random
from sklearn.model_selection import train_test_split

def create_splits(audio_dir="data", out_csv="dataset_splits.csv", test_size=0.2, val_size=0.1):
    """
    Scan all subfolders inside audio_dir (training + test)
    Assign numeric labels to each folder
    Split into train/val/test (stratified)
    """

    # Map folder names to numeric labels automatically
    label_map = {}
    current_label = 1
    data = []

    for root, dirs, files in os.walk(audio_dir):
        # Skip the root folder itself
        if root == audio_dir:
            continue

        folder_name = os.path.basename(root)
        if folder_name not in label_map:
            label_map[folder_name] = current_label
            current_label += 1

        for f in files:
            if f.endswith(".wav"):
                filepath = os.path.join(root, f)
                label = label_map[folder_name]
                data.append((filepath, label))

    if len(data) == 0:
        print("❌ No .wav files found in", audio_dir)
        return

    # Shuffle and split
    random.shuffle(data)
    filepaths = [x[0] for x in data]
    labels = [x[1] for x in data]

    # Compute split sizes
    X_train, X_temp, y_train, y_temp = train_test_split(
        filepaths, labels, test_size=(test_size + val_size), stratify=labels, random_state=42
    )
    relative_val = val_size / (test_size + val_size)
    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=relative_val, stratify=y_temp, random_state=42
    )

    # Write CSV
    with open(out_csv, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["filepath", "label", "split"])
        for x, y in zip(X_train, y_train):
            writer.writerow([x, y, "train"])
        for x, y in zip(X_val, y_val):
            writer.writerow([x, y, "val"])
        for x, y in zip(X_test, y_test):
            writer.writerow([x, y, "test"])

    print(f"✅ dataset_splits.csv saved! Total samples: {len(data)}")
    print("Folder to label mapping:", label_map)

if __name__ == "__main__":
    create_splits()


Overwriting data_loader.py


In [ ]:
from data_loader import create_splits

# Generate dataset_splits.csv from your training and test folders
create_splits(audio_dir="data", out_csv="dataset_splits.csv")


⚠️ Skipping ID013_phonationI.wav (no label found 1–5)
⚠️ Skipping ID007_phonationI.wav (no label found 1–5)
⚠️ Skipping ID001_phonationI.wav (no label found 1–5)
⚠️ Skipping ID017_phonationI.wav (no label found 1–5)
⚠️ Skipping ID018_phonationI.wav (no label found 1–5)
⚠️ Skipping ID015_phonationI.wav (no label found 1–5)
⚠️ Skipping ID006_phonationI.wav (no label found 1–5)
⚠️ Skipping ID005_phonationI.wav (no label found 1–5)
⚠️ Skipping ID000_phonationI.wav (no label found 1–5)
⚠️ Skipping ID041_phonationI.wav (no label found 1–5)
⚠️ Skipping ID037_phonationI.wav (no label found 1–5)
⚠️ Skipping ID040_phonationI.wav (no label found 1–5)
⚠️ Skipping ID033_phonationI.wav (no label found 1–5)
⚠️ Skipping ID035_phonationI.wav (no label found 1–5)
⚠️ Skipping ID029_phonationI.wav (no label found 1–5)
⚠️ Skipping ID038_phonationI.wav (no label found 1–5)
⚠️ Skipping ID010_phonationI.wav (no label found 1–5)
⚠️ Skipping ID016_phonationI.wav (no label found 1–5)
⚠️ Skipping ID026_phonationI

In [ ]:
import importlib
import data_loader
importlib.reload(data_loader)


<module 'data_loader' from '/content/drive/MyDrive/ALS_project/data_loader.py'>

In [ ]:
import data_loader
data_loader.create_splits(audio_dir="data", out_csv="dataset_splits.csv")


✅ dataset_splits.csv saved! Total samples: 2712
Folder to label mapping: {'training': 1, 'phonationI': 2, 'rhythmPA': 3, 'phonationO': 4, 'phonationA': 5, 'phonationU': 6, 'rhythmTA': 7, 'phonationE': 8, 'rhythmKA': 9, 'test': 10}


In [ ]:
%%writefile data_loader.py
import os
import csv
import random
from sklearn.model_selection import train_test_split

def create_splits(audio_dir="data", out_csv="dataset_splits.csv", test_size=0.2, val_size=0.1):
    """
    Scan all subfolders inside audio_dir (training + test)
    Assign numeric labels to each subfolder (skip top-level folders)
    Split into train/val/test (stratified)
    """

    label_map = {}
    current_label = 1
    data = []

    # Only look one level deep (subfolders of training/test)
    for top_folder in os.listdir(audio_dir):
        top_path = os.path.join(audio_dir, top_folder)
        if not os.path.isdir(top_path):
            continue
        for subfolder in os.listdir(top_path):
            subfolder_path = os.path.join(top_path, subfolder)
            if not os.path.isdir(subfolder_path):
                continue
            if subfolder not in label_map:
                label_map[subfolder] = current_label
                current_label += 1
            for f in os.listdir(subfolder_path):
                if f.endswith(".wav"):
                    filepath = os.path.join(subfolder_path, f)
                    label = label_map[subfolder]


Overwriting data_loader.py


In [ ]:
import importlib
import data_loader
importlib.reload(data_loader)



In [ ]:
data_loader.create_splits(audio_dir="data", out_csv="dataset_splits.csv")


In [ ]:
%%writefile preprocess.py
import os
import librosa
import numpy as np
import torch
import pandas as pd
from tqdm import tqdm

def extract_mfcc(filepath, n_mfcc=40, max_len=100):
    """
    Extract MFCC features from a .wav file
    Pads or truncates to max_len frames
    """
    y, sr = librosa.load(filepath, sr=16000)
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    # transpose to (time, n_mfcc)
    mfcc = mfcc.T
    # pad or truncate
    if len(mfcc) < max_len:
        pad_width = max_len - len(mfcc)
        mfcc = np.pad(mfcc, ((0, pad_width), (0,0)), mode='constant')
    else:
        mfcc = mfcc[:max_len, :]
    return mfcc

def preprocess_dataset(csv_file="dataset_splits.csv", save_dir="processed_data", n_mfcc=40, max_len=100):
    os.makedirs(save_dir, exist_ok=True)
    df = pd.read_csv(csv_file)

    X_train, y_train = [], []
    X_val, y_val = [], []
    X_test, y_test = [], []

    for idx, row in tqdm(df.iterrows(), total=len(df)):
        mfcc = extract_mfcc(row['filepath'], n_mfcc=n_mfcc, max_len=max_len)
        label = row['label']
        split = row['split']

        if split == "train":
            X_train.append(mfcc)
            y_train.append(label)
        elif split == "val":
            X_val.append(mfcc)
            y_val.append(label)
        else:
            X_test.append(mfcc)
            y_test.append(label)

    # convert to torch tensors
    X_train = torch.tensor(np.array(X_train), dtype=torch.float32)
    y_train = torch.tensor(np.array(y_train)-1, dtype=torch.long)  # 0-based labels
    X_val = torch.tensor(np.array(X_val), dtype=torch.float32)
    y_val = torch.tensor(np.array(y_val)-1, dtype=torch.long)
    X_test = torch.tensor(np.array(X_test), dtype=torch.float32)
    y_test = torch.tensor(np.array(y_test)-1, dtype=torch.long)

    # save tensors
    torch.save((X_train, y_train), os.path.join(save_dir, "train.pt"))
    torch.save((X_val, y_val), os.path.join(save_dir, "val.pt"))
    torch.save((X_test, y_test), os.path.join(save_dir, "test.pt"))

    print(f"✅ Preprocessed data saved in {save_dir}/")

if __name__ == "__main__":
    preprocess_dataset()


Writing preprocess.py


In [ ]:
import os
os.getcwd()
!ls



data		dataset_splits.csv  preprocess.py  results
data_loader.py	plots		    __pycache__


In [ ]:
import importlib
import preprocess
importlib.reload(preprocess)

preprocess.preprocess_dataset(csv_file="dataset_splits.csv", save_dir="processed_data")


100%|██████████| 2712/2712 [15:36<00:00,  2.89it/s]


✅ Preprocessed data saved in processed_data/


In [ ]:
%%writefile model_baseline_A.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class CNN_MFCC(nn.Module):
    def __init__(self, n_classes=9, n_mfcc=40, max_len=100):
        super(CNN_MFCC, self).__init__()
        self.conv1 = nn.Conv2d(1, 16, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(16)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(32)
        self.fc1 = nn.Linear(32 * (max_len//4) * (n_mfcc//4), 128)
        self.dropout = nn.Dropout(0.3)
        self.fc2 = nn.Linear(128, n_classes)

    def forward(self, x):
        # x shape: (batch, time, n_mfcc)
        x = x.unsqueeze(1)  # add channel: (batch, 1, time, n_mfcc)
        x = self.pool(F.relu(self.bn1(self.conv1(x))))
        x = self.pool(F.relu(self.bn2(self.conv2(x))))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

if __name__ == "__main__":
    # test model shape
    model = CNN_MFCC()
    x = torch.randn(2, 100, 40)  # batch of 2 samples
    y = model(x)
    print(y.shape)  # should be (2, 9)


Writing model_baseline_A.py


In [ ]:
%%writefile train.py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from model_baseline_A import CNN_MFCC
import os

# Paths
data_dir = "processed_data"
save_dir = "results"
os.makedirs(save_dir, exist_ok=True)

# Hyperparameters
batch_size = 32
learning_rate = 0.001
epochs = 30

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load preprocessed tensors
X_train, y_train = torch.load(os.path.join(data_dir, "train.pt"))
X_val, y_val = torch.load(os.path.join(data_dir, "val.pt"))

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Model
n_classes = len(torch.unique(y_train))
model = CNN_MFCC(n_classes=n_classes)
model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
best_val_acc = 0.0
for epoch in range(1, epochs+1):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
        _, predicted = outputs.max(1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

    train_loss = running_loss / total
    train_acc = correct / total

    #


Writing train.py


In [ ]:
!python train.py


Using device: cpu
Traceback (most recent call last):
  File "/content/drive/MyDrive/ALS_project/train.py", line 51, in <module>
    loss = criterion(outputs, y_batch)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1773, in _wrapped_call_impl
    return self._call_impl(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/module.py", line 1784, in _call_impl
    return forward_call(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/modules/loss.py", line 1310, in forward
    return F.cross_entropy(
           ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py", line 3462, in cross_entropy
    return torch._C._nn.cross_entropy_loss(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
IndexError: Target 8 is out of bounds.


In [ ]:
# Overwrite train.py with the fix
%%writefile train.py
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from model_baseline_A import CNN_MFCC
import os

# Paths
data_dir = "processed_data"
save_dir = "results"
os.makedirs(save_dir, exist_ok=True)

# Hyperparameters
batch_size = 32
learning_rate = 0.001
epochs = 30

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Load preprocessed tensors
X_train, y_train = torch.load(os.path.join(data_dir, "train.pt"))
X_val, y_val = torch.load(os.path.join(data_dir, "val.pt"))

train_dataset = TensorDataset(X_train, y_train)
val_dataset = TensorDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

# Fixed number of classes
n_classes = 9  # total number of phonation/rhythm subfolders

# Model
model = CNN_MFCC(n_classes=n_classes)
model.to(device)

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

# Training loop
best_val_acc = 0.0
for epoch in range(1, epochs+1):
    model.train()
    running_loss = 0.0
    correct, total = 0, 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * X_batch.size(0)
        _, predicted = outputs.max(1)
        total += y_batch.size(0)
        correct += (predicted == y_batch).sum().item()

    train_loss = running_loss / total
    train_acc = correct / total

    # Validation
    model.eval()
    correct_val, total_val = 0, 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)
            _, predicted = outputs.max(1)
            total_val += y_batch.size(0)
            correct_val += (predicted == y_batch).sum().item()
    val_acc = correct_val / total_val

    print(f"Epoch {epoch}/{epochs}: Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}, Val Acc: {val_acc:.4f}")

    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), os.path.join(save_dir, "cnn_best.pt"))

print("✅ Training complete. Best val acc:", best_val_acc)


Overwriting train.py


In [ ]:
!python train.py


Using device: cpu
Epoch 1/30: Train Loss: 1.5098, Train Acc: 0.4600, Val Acc: 0.6882
Epoch 2/30: Train Loss: 0.8257, Train Acc: 0.6786, Val Acc: 0.7583
Epoch 3/30: Train Loss: 0.6059, Train Acc: 0.7703, Val Acc: 0.7989
Epoch 4/30: Train Loss: 0.5453, Train Acc: 0.7887, Val Acc: 0.8266
Epoch 5/30: Train Loss: 0.4875, Train Acc: 0.8098, Val Acc: 0.8192
Epoch 6/30: Train Loss: 0.4727, Train Acc: 0.8224, Val Acc: 0.8155
Epoch 7/30: Train Loss: 0.4208, Train Acc: 0.8504, Val Acc: 0.8413
Epoch 8/30: Train Loss: 0.3743, Train Acc: 0.8472, Val Acc: 0.8358
Epoch 9/30: Train Loss: 0.3893, Train Acc: 0.8556, Val Acc: 0.7804
Epoch 10/30: Train Loss: 0.3327, Train Acc: 0.8751, Val Acc: 0.8450
Epoch 11/30: Train Loss: 0.3016, Train Acc: 0.8857, Val Acc: 0.8376
Epoch 12/30: Train Loss: 0.2863, Train Acc: 0.8867, Val Acc: 0.8432
Epoch 13/30: Train Loss: 0.2653, Train Acc: 0.8962, Val Acc: 0.8579
Epoch 14/30: Train Loss: 0.2835, Train Acc: 0.8978, Val Acc: 0.8376
Epoch 15/30: Train Loss: 0.2299, Train 

In [ ]:
%%writefile evaluate.py
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from model_baseline_A import CNN_MFCC
import os
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

# Paths
data_dir = "processed_data"
model_path = "results/cnn_best.pt"
results_dir = "results"
os.makedirs(results_dir, exist_ok=True)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load test data
X_test, y_test = torch.load(os.path.join(data_dir, "test.pt"))
test_dataset = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

# Load model
n_classes = 9
model = CNN_MFCC(n_classes=n_classes)
model.load_state_dict(torch.load(model_path, map_location=device))
model.to(device)
model.eval()

# Predict
all_preds, all_labels = [], []
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(y_batch.numpy())

# Metrics
acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds, average='macro')
cm = confusion_matrix(all_labels, all_preds)

print(f"Test Accuracy: {acc:.4f}")
print(f"Test Macro F1-score: {f1:.4f}")

# Save metrics
metrics_df = pd.DataFrame({"Accuracy": [acc], "Macro_F1": [f1]})
metrics_df.to_csv(os.path.join(results_dir, "metrics.csv"), index=False)

# Plot confusion matrix
plt.figure(figsize=(8,6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Confusion Matrix")
plt.savefig(os.path.join(results_dir, "confusion_matrix.png"))
plt.show()

print(f"✅ Evaluation complete. Metrics and confusion matrix saved in {results_dir}/")


Writing evaluate.py


In [ ]:
!python evaluate.py


Test Accuracy: 0.9007
Test Macro F1-score: 0.8991
Figure(800x600)
✅ Evaluation complete. Metrics and confusion matrix saved in results/
